# Chapter 12 — Exercise Solutions## When to Use RL for Agents[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 12.1: Extended Decision Framework

In [ ]:
# Full 6-question RL decision framework with concrete recommendationsdef rl_decision_framework(    has_success_signal: bool,    is_sequential: bool,       # >10 steps    tried_simpler: bool,       # SFT + DPO attempted    safe_environment: bool,    engineering_resources: bool,    task_value_high: bool,    is_verifiable: bool = False) -> dict:    """    Returns: recommendation, algorithm, reasoning    """    if not has_success_signal:        return {            'recommendation': 'SFT + Prompting Only',            'algorithm': 'None (no RL)',            'reason': 'Cannot measure success → RL will optimise toward the wrong thing',            'risk': 'HIGH — unmeasurable reward leads to reward hacking guaranteed'        }    if not is_sequential:        return {            'recommendation': 'DPO or Contextual Bandit',            'algorithm': 'DPO for preference alignment; UCB/Thompson for single-step optimisation',            'reason': 'Single-step task — full RL (PPO/GRPO) is massive overkill',            'risk': 'LOW'        }    if not tried_simpler:        return {            'recommendation': 'Try SFT → DPO → evaluate → THEN consider RL',            'algorithm': 'SFT first, then DPO',            'reason': 'Premature RL: always try simpler methods before committing to RL training',            'risk': 'MEDIUM — skipping simpler methods wastes compute and time'        }    if not safe_environment:        return {            'recommendation': 'Offline RL (CQL, Decision Transformer, IQL)',            'algorithm': 'Conservative Q-Learning or Decision Transformer',            'reason': 'No safe training env → cannot do online exploration → offline RL only',            'risk': 'HIGH — online RL in unsafe env can cause real-world harm'        }    if not engineering_resources:        return {            'recommendation': 'Use frontier model API + strong prompting',            'algorithm': 'Zero-shot or few-shot prompting with GPT-4o/Claude',            'reason': 'RL requires significant engineering infrastructure — use pre-trained RL',            'risk': 'LOW'        }    if not task_value_high:        return {            'recommendation': 'Accept current performance ceiling',            'algorithm': 'Best prompted/SFT system',            'reason': 'ROI does not justify RL training cost for this task',            'risk': 'NONE'        }    # All criteria met — now choose the algorithm    if is_verifiable:        return {            'recommendation': 'GRPO + RLVR',            'algorithm': 'GRPO with verifiable reward (math/code/SQL verifier)',            'reason': 'Verifiable task + all criteria → GRPO is simpler and more efficient than PPO',            'risk': 'LOW-MEDIUM — cold start if model has <5% initial accuracy'        }    else:        return {            'recommendation': 'PPO-RLHF with reward model',            'algorithm': 'Bradley-Terry reward model + PPO (TRL PPOTrainer)',            'reason': 'Non-verifiable subjective task + all criteria → need learned reward model',            'risk': 'MEDIUM — reward model quality determines training quality'        }# Test with different scenariosscenarios = [    {'name': 'SQL generation (verifiable)',     'args': dict(has_success_signal=True, is_sequential=False, tried_simpler=True,                  safe_environment=True, engineering_resources=True,                  task_value_high=True, is_verifiable=True)},    {'name': 'Customer service chatbot (subjective)',     'args': dict(has_success_signal=True, is_sequential=True, tried_simpler=True,                  safe_environment=True, engineering_resources=True,                  task_value_high=True, is_verifiable=False)},    {'name': 'Drug discovery agent (unsafe env)',     'args': dict(has_success_signal=True, is_sequential=True, tried_simpler=True,                  safe_environment=False, engineering_resources=True,                  task_value_high=True, is_verifiable=False)},    {'name': 'Email subject line optimisation',     'args': dict(has_success_signal=True, is_sequential=False, tried_simpler=True,                  safe_environment=True, engineering_resources=True,                  task_value_high=True, is_verifiable=False)},]for s in scenarios:    result = rl_decision_framework(**s['args'])    print(f"\n{s['name']}")    print(f"  Recommendation: {result['recommendation']}")    print(f"  Algorithm:      {result['algorithm']}")    print(f"  Risk level:     {result['risk']}")

### Solution 12.2: ROI Calculator

In [ ]:
def rl_roi_analysis(task_volume_per_day, current_success_rate,                    expected_improvement, value_per_success_usd,                    training_compute_cost_usd, ongoing_cost_per_day_usd=0):    """    Full ROI analysis for RL training investment.    Returns: breakeven_days, 1yr_roi, recommendation    """    value_before = task_volume_per_day * current_success_rate * value_per_success_usd    value_after  = task_volume_per_day * (current_success_rate + expected_improvement) * value_per_success_usd    daily_gain   = value_after - value_before - ongoing_cost_per_day_usd    if daily_gain <= 0:        return {'breakeven': float('inf'), '1yr_roi': -100, 'recommendation': 'Do NOT invest — negative daily gain'}    breakeven_days = training_compute_cost_usd / daily_gain    profit_1yr     = (daily_gain * 365) - training_compute_cost_usd    roi_1yr        = profit_1yr / training_compute_cost_usd * 100    rec = ('Strong YES' if breakeven_days < 30  else           'YES'        if breakeven_days < 90  else           'MARGINAL'   if breakeven_days < 180 else 'NO — too slow')    return {        'daily_value_before':  round(value_before, 2),        'daily_value_after':   round(value_after, 2),        'daily_gain':          round(daily_gain, 2),        'breakeven_days':      round(breakeven_days, 1),        'profit_1yr':          round(profit_1yr, 2),        'roi_1yr_pct':         round(roi_1yr, 1),        'recommendation':      rec    }# Case studiescases = [    {'name': 'SQL coding assistant',     'task_volume_per_day': 5000, 'current_success_rate': 0.73,     'expected_improvement': 0.08, 'value_per_success_usd': 0.50,     'training_compute_cost_usd': 2000, 'ongoing_cost_per_day_usd': 5},    {'name': 'Customer service bot',     'task_volume_per_day': 10000, 'current_success_rate': 0.65,     'expected_improvement': 0.12, 'value_per_success_usd': 2.00,     'training_compute_cost_usd': 8000, 'ongoing_cost_per_day_usd': 20},    {'name': 'Internal document search',     'task_volume_per_day': 200, 'current_success_rate': 0.80,     'expected_improvement': 0.05, 'value_per_success_usd': 1.00,     'training_compute_cost_usd': 3000, 'ongoing_cost_per_day_usd': 0},]for case in cases:    name = case.pop('name')    result = rl_roi_analysis(**case)    print(f"\n{'='*50}")    print(f"Case: {name}")    print(f"  Daily gain:      ${result['daily_gain']:.2f}")    print(f"  Breakeven:       {result['breakeven_days']:.0f} days")    print(f"  1-year ROI:      {result['roi_1yr_pct']:.0f}%")    print(f"  Recommendation:  {result['recommendation']}")